# API-based cleaning (Information Systems)

Same logic as root `datacleaning.py`: unique (Author_name, Author_Address) pairs, batched API calls, cache, 3 RPM / 200 RPD.

In [ ]:
import os
import json
import time
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

INPUT_CSV = Path("Information_Systems_article_data.csv")
OUTPUT_CSV = Path("Information_Systems_API_cleaned.csv")
CACHE_JSON = Path("standardization_cache.json")
RPM_LIMIT = 3
RPD_LIMIT = 200
DELAY = 60.0 / RPM_LIMIT
BATCH_SIZE = 10

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
MODEL = "gpt-4o-mini"

data = pd.read_csv(INPUT_CSV)
total_rows = len(data)

def _cell(x):
    if x is None or (isinstance(x, float) and pd.isna(x)): return ""
    s = str(x).strip()
    return "" if s == "nan" else s

def _pair(row):
    return (_cell(row.get("Author_name", "")), _cell(row.get("Author_Address", "")))

def pair_to_text(pair):
    return " ".join(p for p in pair if p).strip()

SYSTEM_PROMPT = """You extract and standardize author/university/location from text.
Return ONLY valid JSON. For each input text return one object in the \"results\" array, in the SAME ORDER.
Each object: Standardized_Author, University, Department, State, Country, Pincode (use \"\" or null if missing).
Rules: English only, title case names, full university names (no abbreviations), full country names."""

print("Config loaded.")

In [ ]:
def extract_batch(texts):
    if not texts: return []
    numbered = "\n".join(f"{i+1}. {t}" for i, t in enumerate(texts))
    user = f"Process these {len(texts)} texts in order and return a JSON object with key 'results' (array of {len(texts)} objects, each with Standardized_Author, University, Department, State, Country, Pincode):\n\n{numbered}"
    r = client.chat.completions.create(model=MODEL, messages=[{"role":"system","content":SYSTEM_PROMPT},{"role":"user","content":user}], temperature=0, response_format={"type":"json_object"})
    raw = r.choices[0].message.content.strip().strip("```json").strip("```")
    try:
        out = json.loads(raw)
        results = out.get("results", [])
    except Exception as e:
        print("Parse error:", e)
        results = []
    def _v(x): return x if x is not None and str(x).strip() else ""
    out_list = []
    for i in range(len(texts)):
        res = results[i] if i < len(results) else {}
        out_list.append({"Standardized_Author":_v(res.get("Standardized_Author")),"University":_v(res.get("University")),"Department":_v(res.get("Department")),"State":_v(res.get("State")),"Country":_v(res.get("Country")),"Pincode":_v(res.get("Pincode"))})
    return out_list

In [ ]:
# Unique pairs and cache (preview: run this to see how many rows/API calls)
rows_by_pair = {}
for i, row in data.iterrows():
    p = _pair(row)
    if p not in rows_by_pair:
        rows_by_pair[p] = []
    rows_by_pair[p].append(i)
unique_pairs = list(rows_by_pair.keys())
pairs_needing_api = [p for p in unique_pairs if pair_to_text(p)]

cache = {}
if CACHE_JSON.exists():
    with open(CACHE_JSON, "r", encoding="utf-8") as f:
        cache = {tuple(k): v for k, v in json.load(f).items()}
    print(f"Cache: {len(cache)} pairs already done.")

pending = [p for p in pairs_needing_api if p not in cache]
max_pairs_run = RPD_LIMIT * BATCH_SIZE
to_fetch = pending[:max_pairs_run]
num_calls = (len(to_fetch) + BATCH_SIZE - 1) // BATCH_SIZE

print(f"Total rows: {total_rows}. Unique pairs (non-empty): {len(pairs_needing_api)}.")
print(f"Pending: {len(pending)}. This run would: {len(to_fetch)} pairs in {num_calls} API call(s).")

In [ ]:
# Call API for to_fetch (batches of BATCH_SIZE)
for start in range(0, len(to_fetch), BATCH_SIZE):
    batch_pairs = to_fetch[start : start + BATCH_SIZE]
    texts = [pair_to_text(p) for p in batch_pairs]
    try:
        results = extract_batch(texts)
        for p, res in zip(batch_pairs, results):
            cache[p] = res
        print(f"Cached {start + len(batch_pairs)}/{len(to_fetch)}")
    except Exception as e:
        print("Error:", e)
        for p in batch_pairs:
            cache[p] = {"Standardized_Author": p[0], "University": "", "Department": "", "State": "", "Country": "", "Pincode": ""}
    time.sleep(DELAY)

with open(CACHE_JSON, "w", encoding="utf-8") as f:
    json.dump({list(k): v for k, v in cache.items()}, f, indent=0, ensure_ascii=False)
print("Cache saved.")

In [ ]:
# Fill cache for empty pairs; then write full output CSV
for p in unique_pairs:
    if p not in cache:
        cache[p] = {"Standardized_Author": p[0], "University": "", "Department": "", "State": "", "Country": "", "Pincode": ""}

HEADER = ["URL","Journal_Title","Article_Title","Volume_Issue","Month_Year","Abstract","Keywords","Author_name","Standardized_Author","Author_email","Author_Address","Standardized_University","Author_Department","Author_State","Author_Country","Author_Pincode"]
with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
    import csv
    w = csv.writer(f)
    w.writerow(HEADER)
    for i in range(total_rows):
        row = data.iloc[i]
        p = _pair(row)
        res = cache.get(p, {})
        if not res:
            res = {"Standardized_Author": _cell(row.get("Author_name")), "University": "", "Department": "", "State": "", "Country": "", "Pincode": ""}
        w.writerow([_cell(row.get("URL")), _cell(row.get("Journal_Title")), _cell(row.get("Article_Title")), _cell(row.get("Volume_Issue")), _cell(row.get("Month_Year")), _cell(row.get("Abstract")), _cell(row.get("Keywords")), _cell(row.get("Author_name")), res.get("Standardized_Author",""), _cell(row.get("Author_email")), _cell(row.get("Author_Address")), res.get("University",""), res.get("Department",""), res.get("State",""), res.get("Country",""), res.get("Pincode","")])
print(f"Wrote {total_rows} rows to {OUTPUT_CSV}.")